### Connect to Betfair

Import the libraries, read login details from `credentials.json`, and create and log in to the Betfair API client. The final line displays the session token.


In [ ]:
import betfairlightweight
from betfairlightweight import filters
import pandas as pd
import numpy as np
import os
import datetime
import json

with open('credentials.json') as f:
    creds = json.load(f)
    my_username = creds['username']
    my_password = creds['password']
    my_app_key = creds['app_key']

trading = betfairlightweight.APIClient(my_username, my_password, app_key=my_app_key)

trading.login_interactive()
trading.session_token

: 

### List sports and event types

Fetch the available Betfair event types and show their names and IDs in a table.


In [ ]:
event_types = trading.betting.list_event_types()

sport_ids = pd.DataFrame({'Sport': [event_type_object.event_type.name for event_type_object in event_types],
                          'ID': [event_type_object.event_type.id for event_type_object in event_types]
                          }).sort_index()

sport_ids

### Find politics competitions

Request competitions for event type `2378961` (politics), with a market start-time cutoff one week from now, and build a competition table.


In [ ]:
datetime_in_a_week = (datetime.datetime.now() + datetime.timedelta(weeks=1)).strftime("%Y-%m-%dT%TZ")

competition_filter = betfairlightweight.filters.market_filter(
    event_type_ids=[2378961],
    market_start_time={
        'to': datetime_in_a_week
    })

competitions = trading.betting.list_competitions(filter=competition_filter)

politics_competitions = pd.DataFrame({'Competition': [competition_object.competition.name for competition_object in competitions],
                                      'ID': [competition_object.competition.id for competition_object in competitions]
                                      })

### View competitions

Display the competitions returned by the previous cell.


In [ ]:
politics_competitions

### Find midterm election markets

Look up the USA 2026 Midterm Elections competition and list up to 1,000 of its markets, including market IDs and event names.


In [ ]:
midterms_competition_id = politics_competitions.loc[politics_competitions['Competition'] == 'USA 2026 Midterm Elections', 'ID'].values[0]
market_filter = betfairlightweight.filters.market_filter(
    event_type_ids=['2378961'],
    competition_ids=[midterms_competition_id]
)

markets = trading.betting.list_market_catalogue(
    filter=market_filter,
    market_projection=[
        'RUNNER_DESCRIPTION',
        'MARKET_START_TIME',
        'EVENT'
    ],
    max_results=1000
)

markets_df = pd.DataFrame({
    'Market': [market.market_name for market in markets],
    'Market ID': [market.market_id for market in markets],
    'Event': [
        market.event.name if market.event else None
        for market in markets
    ]
})

markets_df


### Refresh the login

Log in again to refresh the Betfair session. The final line displays the session token.


In [ ]:
trading.login_interactive()
trading.session_token

### Check historical data access

Use the session token to request historical data available to the account and display the response as a table. Stop with an error if the request fails.


In [ ]:
import requests
import pandas as pd
import os

historical_url = "https://historicdata.betfair.com/api/GetMyData"

headers = {
    "ssoid": trading.session_token
}

response = requests.get(historical_url, headers=headers)

response.raise_for_status()

my_historical_data = pd.DataFrame(response.json())

my_historical_data

### Explore historical data options

Request the available collection options for Other Sports on the Basic Plan during February 2026, then print the response.


In [ ]:
historical_url = "https://historicdata.betfair.com/api/"

headers = {
    "ssoid": trading.session_token
}

filter_options = {
    "sport": "Other Sports",
    "plan": "Basic Plan",
    "fromDay": 1,
    "fromMonth": 2,
    "fromYear": 2026,
    "toDay": 28,
    "toMonth": 2,
    "toYear": 2026,
    "eventId": None,
    "eventName": None,
    "marketTypesCollection": [],
    "countriesCollection": [],
    "fileTypeCollection": ["M"]
}

response = requests.post(
    historical_url + "GetCollectionOptions",
    headers=headers,
    json=filter_options
)

response.raise_for_status()

options = response.json()

print(options)

### Compare available market types

Repeat the February options request and display market types in descending order of their returned counts.


In [ ]:
sport = "Other Sports"
plan = "Basic Plan"

filter_options = {
    "sport": sport,
    "plan": plan,
    "fromDay": 1,
    "fromMonth": 2,
    "fromYear": 2026,
    "toDay": 28,
    "toMonth": 2,
    "toYear": 2026,
    "eventId": None,
    "eventName": None,
    "marketTypesCollection": [],
    "countriesCollection": [],
    "fileTypeCollection": ["M"]
}

response = requests.post(
    historical_url + "GetCollectionOptions",
    headers=headers,
    json=filter_options
)

response.raise_for_status()

options = response.json()

market_types = pd.DataFrame(options["marketTypesCollection"])

market_types.sort_values("count", ascending=False)

### View file types

Display the file-type options returned by the previous request.


In [ ]:
pd.DataFrame(options["fileTypeCollection"])

### Find February files

Request a list of February 2026 MATCH_ODDS files matching the event name Ireland. Show the number of files and the first ten paths; this does not download the files yet.


In [ ]:
download_filter = {
    "sport": sport,
    "plan": plan,
    "fromDay": 1,
    "fromMonth": 2,
    "fromYear": 2026,
    "toDay": 28,
    "toMonth": 2,
    "toYear": 2026,
    "eventId": None,
    "eventName": "Ireland",
    "marketTypesCollection": ["MATCH_ODDS"],
    "countriesCollection": [],
    "fileTypeCollection": ["M"]
}

response = requests.post(
    historical_url + "DownloadListOfFiles",
    headers=headers,
    json=download_filter
)

response.raise_for_status()

february_files = response.json()

print("Files found:", len(february_files))

february_files[:10]

### Find March files

Change the same filter to March 2026 and request the matching file paths, retaining the Ireland and MATCH_ODDS filters.


In [ ]:
download_filter.update({
    "fromDay": 1,
    "fromMonth": 3,
    "toDay": 31,
    "toMonth": 3
})

response = requests.post(
    historical_url + "DownloadListOfFiles",
    headers=headers,
    json=download_filter
)

response.raise_for_status()

march_files = response.json()

print("Files found:", len(march_files))

### Create the download folder

Create `six_nations_data` if it does not already exist.


In [ ]:
os.makedirs("six_nations_data", exist_ok=True)

### Download February files

Download each February file and save it in `six_nations_data` using its original filename. Existing files with the same name are overwritten.


In [ ]:
from pathlib import PurePosixPath

for file_path in february_files:

    filename = PurePosixPath(file_path).name

    download_url = historical_url + "DownloadFile"

    file_response = requests.get(
        download_url,
        headers=headers,
        params={"filePath": file_path}
    )

    file_response.raise_for_status()

    with open(f"six_nations_data/{filename}", "wb") as f:
        f.write(file_response.content)

print("February downloads complete!")

### Download March files

Download and save the March files in the same folder. Existing files with the same name are overwritten.


In [ ]:
from pathlib import PurePosixPath

for file_path in march_files:

    filename = PurePosixPath(file_path).name

    download_url = historical_url + "DownloadFile"

    file_response = requests.get(
        download_url,
        headers=headers,
        params={"filePath": file_path}
    )

    file_response.raise_for_status()

    with open(f"six_nations_data/{filename}", "wb") as f:
        f.write(file_response.content)

print("March downloads complete!")

### Load file-reading libraries

Import the tools used to open compressed files, parse JSON messages, find local files, and build tables.


In [ ]:
import bz2
import json
import os
import glob
import pandas as pd

### Find downloaded files

Collect the `.bz2` files in `six_nations_data`, print their count, and preview the first ten paths.


In [ ]:
data_folder = "six_nations_data"

files = glob.glob(os.path.join(data_folder, "*.bz2"))

print("Number of files:", len(files))

files[:10]

### Inspect a sample message

Open the first compressed file and print up to 4,000 characters of its first JSON message to inspect the data structure. This requires at least one file.


In [ ]:
# Open the first downloaded file
sample_file = files[0]

with bz2.open(sample_file, "rt") as f:

    for i, line in enumerate(f):

        data = json.loads(line)

        print(json.dumps(data, indent=2)[:4000])

        if i == 0:
            break

### Define a market summary helper

Define a function that reads a file until it finds a market definition, then returns its market ID, event, scheduled start time, and runner names.


In [ ]:
def get_market_info(file_path):

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            for market_change in data.get("mc", []):

                market_definition = market_change.get(
                    "marketDefinition"
                )

                if market_definition:

                    runners = market_definition.get("runners", [])

                    return {
                        "File": os.path.basename(file_path),
                        "Market ID": market_change.get("id"),
                        "Event": market_definition.get("eventName"),
                        "Market": market_definition.get("name"),
                        "Market Type": market_definition.get("marketType"),
                        "Kick-off": market_definition.get("marketTime"),
                        "Runners": [
                            runner.get("name")
                            for runner in runners
                        ]
                    }

    return None

### Build the market summary table

Apply the helper to every downloaded file and combine the available market summaries into a table. This replaces the earlier `markets_df` variable.


In [ ]:
market_information = []

for file_path in files:

    market_info = get_market_info(file_path)

    if market_info is not None:
        market_information.append(market_info)

markets_df = pd.DataFrame(market_information)

markets_df.head()

### Show Ireland markets

Keep rows whose event name contains Ireland, ignoring letter case and missing names.


In [ ]:
ireland_markets = markets_df[
    markets_df["Event"].str.contains(
        "Ireland",
        case=False,
        na=False
    )
]

ireland_markets

### Calculate the target time

Convert scheduled start times to UTC and calculate the time one hour before each start.


In [ ]:
# Get the scheduled kick-off time
kick_off = pd.to_datetime(
    markets_df["Kick-off"],
    utc=True
)

# Calculate one hour before kick-off
target_time = kick_off - pd.Timedelta(hours=1)

### Define a historical price helper

Read messages at or before a supplied target time and retain each runner's latest recorded last traded price. These are last traded prices, which may have been recorded earlier than the target time.


In [ ]:
def get_ltp_at_time(file_path, target_time):

    runner_names = {}
    latest_prices = {}

    target_ms = int(target_time.timestamp() * 1000)

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            timestamp = data.get("pt")

            if timestamp is None or timestamp > target_ms:
                continue

            for mc in data.get("mc", []):

                definition = mc.get("marketDefinition", {})

                for runner in definition.get("runners", []):
                    runner_names[runner["id"]] = runner["name"]

                for runner_change in mc.get("rc", []):

                    selection_id = runner_change["id"]

                    if "ltp" in runner_change:
                        latest_prices[selection_id] = runner_change["ltp"]

    return [
        {
            "Selection": runner_names.get(selection_id),
            "Last Traded Price": price
        }
        for selection_id, price in latest_prices.items()
    ]

### Filter for senior Ireland matches

Select Ireland events and exclude names matching U20, Women, XV, Azerbaijan, or Luxembourg. This is a name-based filter rather than a verified competition classification.


In [ ]:
ireland_senior_matches = markets_df[
    markets_df["Event"].str.contains(
        "Ireland",
        case=False,
        na=False
    )
    &
    ~markets_df["Event"].str.contains(
        "U20|Women|XV|Azerbaijan|Luxembourg",
        case=False,
        na=False
    )
]

ireland_senior_matches[
    ["Event", "Market ID", "Kick-off", "Runners"]
]

### Sort matches and redefine the helper

Sort the selected matches by scheduled start time and reset their row numbers. The cell also repeats the earlier definition of the price helper.


In [ ]:
ireland_senior_matches = ireland_senior_matches.sort_values(
    "Kick-off"
).reset_index(drop=True)

ireland_senior_matches[
    ["Event", "Market ID", "Kick-off"]
]

def get_ltp_at_time(file_path, target_time):

    runner_names = {}
    latest_prices = {}

    target_ms = int(target_time.timestamp() * 1000)

    with bz2.open(file_path, "rt") as f:

        for line in f:

            data = json.loads(line)

            timestamp = data.get("pt")

            if timestamp is None or timestamp > target_ms:
                continue

            for mc in data.get("mc", []):

                definition = mc.get("marketDefinition", {})
    
                for runner in definition.get("runners", []):
                    runner_names[runner["id"]] = runner["name"]

                for runner_change in mc.get("rc", []):

                    selection_id = runner_change["id"]

                    if "ltp" in runner_change:
                        latest_prices[selection_id] = runner_change["ltp"]

    return [
        {
            "Selection": runner_names.get(selection_id),
            "Last Traded Price": price
        }
        for selection_id, price in latest_prices.items()
    ]

### Extract prices before each match

For every selected match, read the latest recorded prices at or before one hour before its scheduled start. Combine the match, times, selections, and prices into `odds_df`.


In [ ]:
results = []

for _, market in ireland_senior_matches.iterrows():

    kick_off = pd.to_datetime(market["Kick-off"], utc=True)

    target_time = kick_off - pd.Timedelta(hours=1)

    file_path = os.path.join(
        data_folder,
        market["File"]
    )

    prices = get_ltp_at_time(file_path, target_time)

    for price in prices:

        results.append({
            "Match": market["Event"],
            "Kick-off": kick_off,
            "Target Time": target_time,
            "Selection": price["Selection"],
            "Last Traded Price": price["Last Traded Price"]
        })

odds_df = pd.DataFrame(results)

odds_df

### Keep Ireland prices

Filter the results to selections named exactly Ireland and reset the row numbers.


In [ ]:
ireland_odds = odds_df[
    odds_df["Selection"] == "Ireland"
].reset_index(drop=True)

ireland_odds